# Notebook 04 — Cost-Asymmetric Verifier Evaluation

*Companion to Ciphero ML interview prep — anchors the anti-fraud / verifier-eval thread.*

When a verifier is judging high-stakes agent actions (policy violations, fraud, jailbreak attempts), the cost of a missed violation (false negative) is rarely equal to the cost of a spurious flag (false positive). This notebook trains two small PyTorch classifiers on a 99/1 imbalanced synthetic dataset — one with **uniform cross-entropy**, one with **weighted CE (pos_weight = 100)** — and shows three things:

1. **AUC-PR** is the right ranking metric for imbalanced data; ROC AUC is misleading.
2. The **default 0.5 decision threshold is almost never optimal** under cost asymmetry.
3. A **cost-ratio sweep** lets you pick the operating point a customer actually wants — even if you trained with uniform CE.

**JD anchor:** Ciphero's verifier surface (Fakespot lineage; anti-fraud DNA) sells calibrated *probabilities* and customer-tunable thresholds, not hard binary labels. This notebook demonstrates the eval methodology, not the production pipeline.

## Setup choices

**Data.** Synthetic via `sklearn.datasets.make_classification` with 99/1 class weights, 20k samples, 20 features (10 informative). Synthetic over public because: (a) the verifier shape — overwhelming majority compliant, rare-but-costly violations — is rarely captured by tabular benchmarks; (b) we want full control over class separation and feature noise so the cost-asymmetry signal isn't muddied by dataset artifacts.

**Model.** 3-layer MLP in PyTorch (`20 → 64 → 32 → 1`, ReLU activations, single logit output, `BCEWithLogitsLoss`). PyTorch over sklearn because the JD bullet on inference pipelines explicitly calls out PyTorch, and the only mechanical difference here is the `pos_weight` argument to BCE — a one-line knob that's surprisingly load-bearing.

**Reproducibility.** Seeds set on `random`, `numpy`, and `torch` (CPU + CUDA). Train/test split is stratified to preserve the 1% positive rate.

> Run with: `pip install -e ".[dev]"` (PyTorch is now in main dependencies).

In [ ]:
from __future__ import annotations

import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
)

# Add src/ to path when running from the notebooks/ directory.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from llm_eval.cost_asymmetric import (  # noqa: E402
    cost_ratio_sweep,
    eval_at_threshold,
    make_imbalanced_synth,
    precision_at_k,
    predict_proba,
    train_mlp,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

data = make_imbalanced_synth(
    n_samples=20_000, weights=(0.99, 0.01), n_features=20, n_informative=10, seed=SEED
)
print(f"X_train: {data['X_train'].shape}, X_test: {data['X_test'].shape}")
print(
    f"train positives: {int(data['y_train'].sum())} "
    f"({data['y_train'].mean():.3%})"
)
print(
    f"test  positives: {int(data['y_test'].sum())} "
    f"({data['y_test'].mean():.3%})"
)

In [ ]:
# Train two models with identical architecture / seed / data — only the loss differs.
print("Training uniform CE...")
model_uniform, hist_uniform = train_mlp(
    data["X_train"], data["y_train"], pos_weight=None, n_epochs=50, batch_size=256, seed=0
)
print(f"  final epoch loss: {hist_uniform[-1]:.4f}")

print("Training weighted CE (pos_weight = 100)...")
model_weighted, hist_weighted = train_mlp(
    data["X_train"], data["y_train"], pos_weight=100.0, n_epochs=50, batch_size=256, seed=0
)
print(f"  final epoch loss: {hist_weighted[-1]:.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(hist_uniform, label="uniform CE", linewidth=2)
ax.plot(hist_weighted, label="weighted CE (pos_weight=100)", linewidth=2, linestyle="--")
ax.set_xlabel("epoch")
ax.set_ylabel("mean BCE loss")
ax.set_title("Training loss — uniform vs weighted cross-entropy")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
probs_uniform = predict_proba(model_uniform, data["X_test"])
probs_weighted = predict_proba(model_weighted, data["X_test"])
y_test = data["y_test"]

def baseline_row(name: str, probs: np.ndarray, labels: np.ndarray) -> dict:
    m05 = eval_at_threshold(probs, labels, threshold=0.5)
    return {
        "model": name,
        "AUC-PR": average_precision_score(labels, probs),
        "ROC AUC": roc_auc_score(labels, probs),
        "P@10": precision_at_k(probs, labels, 10),
        "P@50": precision_at_k(probs, labels, 50),
        "P@100": precision_at_k(probs, labels, 100),
        "P@t=0.5": m05["precision"],
        "R@t=0.5": m05["recall"],
        "FP@t=0.5": m05["fp"],
        "FN@t=0.5": m05["fn"],
    }

baseline = pd.DataFrame(
    [
        baseline_row("uniform CE", probs_uniform, y_test),
        baseline_row("weighted CE", probs_weighted, y_test),
    ]
)
baseline_show = baseline.copy()
for col in ["AUC-PR", "ROC AUC", "P@10", "P@50", "P@100", "P@t=0.5", "R@t=0.5"]:
    baseline_show[col] = baseline_show[col].map(lambda v: f"{v:.3f}")
baseline_show

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, probs, color in [
    ("uniform CE", probs_uniform, "C0"),
    ("weighted CE (pw=100)", probs_weighted, "C1"),
]:
    precision, recall, _ = precision_recall_curve(y_test, probs)
    ap = average_precision_score(y_test, probs)
    ax.plot(recall, precision, label=f"{name}  (AP = {ap:.3f})", linewidth=2, color=color)

baserate = float(y_test.mean())
ax.axhline(baserate, linestyle=":", color="gray", label=f"random baseline = {baserate:.3f}")
ax.set_xlabel("recall")
ax.set_ylabel("precision")
ax.set_title("Precision-recall curves — imbalanced synth (1% positive)")
ax.legend(loc="upper right")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fp_costs = [1.0, 10.0, 100.0]
fn_costs = [1.0, 10.0, 100.0]

sweep_uniform = cost_ratio_sweep(probs_uniform, y_test, fp_costs, fn_costs)
sweep_weighted = cost_ratio_sweep(probs_weighted, y_test, fp_costs, fn_costs)

print("Cost-ratio sweep — uniform CE model:")
display_cols = ["c_fp", "c_fn", "cost_ratio", "threshold", "fp", "fn", "precision", "recall"]
print(
    sweep_uniform[display_cols]
    .sort_values("cost_ratio")
    .to_string(index=False, float_format=lambda v: f"{v:.3f}")
)

print("\nCost-ratio sweep — weighted CE model:")
print(
    sweep_weighted[display_cols]
    .sort_values("cost_ratio")
    .to_string(index=False, float_format=lambda v: f"{v:.3f}")
)

In [ ]:
# Aggregate over distinct cost ratios (each ratio appears multiple times in the
# 3x3 sweep; the threshold depends only on the ratio, not absolute scale).
def agg_by_ratio(sweep: pd.DataFrame) -> pd.DataFrame:
    out = sweep.groupby("cost_ratio", as_index=False)["threshold"].first()
    return out.sort_values("cost_ratio")

agg_uniform = agg_by_ratio(sweep_uniform)
agg_weighted = agg_by_ratio(sweep_weighted)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(
    agg_uniform["cost_ratio"], agg_uniform["threshold"],
    marker="o", linewidth=2, label="uniform CE",
)
ax.plot(
    agg_weighted["cost_ratio"], agg_weighted["threshold"],
    marker="s", linewidth=2, linestyle="--", label="weighted CE (pw=100)",
)
ax.axhline(0.5, linestyle=":", color="gray", label="default t = 0.5")
ax.set_xscale("log")
ax.set_xlabel("cost ratio  (c_fn / c_fp)")
ax.set_ylabel("optimal threshold  t*")
ax.set_title("Optimal threshold drops as missing a positive grows costlier")
ax.legend()
ax.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()

## Results & takeaway

| Finding | Detail |
|---|---|
| **Ranking** | Uniform CE achieved AUC-PR ≈ 0.56 vs weighted CE ≈ 0.49 — pos-weighting *hurt* ranking quality because it flattens the score distribution. |
| **Default threshold** | At t = 0.5, weighted CE shows precision ≈ 0.07 (catastrophic) while uniform CE shows ≈ 0.90 — but uniform CE only catches 46% of positives. Neither is shippable as-is. |
| **Cost-aware threshold** | Sweeping (c_fp, c_fn) ∈ {1, 10, 100}² and picking the empirically optimal threshold per pair recovers sensible operating points for both models. |
| **Asymmetry signal** | When c_fn = 100 × c_fp, t* drops to ~0.16 (weighted) and ~0.012 (uniform) — both well below the default 0.5. Threshold *must* be tuned downstream of training. |

**Conclusion (≤150 words).** The interview-grade lesson is that loss-weighting and threshold-selection are two distinct knobs solving different problems. Uniform CE plus a downstream cost-aware threshold sweep is generally cleaner than weighted CE: it preserves ranking quality (AUC-PR), and the threshold is the customer-facing dial anyway. Weighted CE is only the right tool when you cannot do post-hoc threshold tuning — e.g., when the model output feeds directly into a hard-coded 0.5 cutoff you don't control. For verifier-shaped problems (rare, expensive false negatives), the production recipe is: train with uniform CE, calibrate probabilities (Notebook 01), expose the threshold as a tunable parameter, and let the customer's policy fix the cost ratio. Prompt-design for LLM judges follows the same pattern — calibrate first, then pick the operating point per use case.

## Open question for Ciphero

> **Where does Ciphero set the FP / FN cost ratio for production verifier output? Customer-tunable or policy-fixed?**

Supporting probes for the conversation:

- *Per-customer policy.* If different enterprise customers have different stomachs for false alarms (e.g., compliance vs marketing-content moderation), the cost ratio is a per-tenant config — and the verifier needs calibrated probabilities, not labels, so the threshold can be set downstream.
- *Asymmetric retraining loops.* If high-cost FNs are sampled into a human-review queue and FPs are silently absorbed, the operating point shifts as the labeled-data distribution shifts. How does Ciphero detect that drift and avoid threshold creep?
- *Default vs override.* Even with customer-tunable thresholds, the SDK ships a default. What's the default cost ratio implied by the shipped threshold, and how is that number defended in customer conversations? (This is the "show your math" question — the ratio implied by t = 0.5 on a calibrated model is a customer-facing claim.)